In [13]:
import pandas as pd
import numpy as np
import glob

In [14]:
df = pd.read_csv('altered_ballots.csv')
df.head()

,ballot_id,1,2,3,4,5
0,0,1,4,NaN,NaN,NaN
1,1,1,4,2.0,3.0,0.0
2,2,1,4,0.0,2.0,3.0
3,3,1,0,4.0,2.0,3.0
4,4,4,2,3.0,1.0,0.0


In [15]:
rank_cols = ['1', '2', '3', '4', '5']

print("Total rows in df:", len(df))
print()
print("Per-column NaN counts:")
print(df[rank_cols].isna().sum())
print()
print("Rows with ALL 5 filled (your original filter):", df[rank_cols].notna().all(axis=1).sum())
print("Rows with AT LEAST 1 filled (the new filter):", df[rank_cols].notna().any(axis=1).sum())
print()
print("First 10 rows of rank_cols, raw:")
print(df[rank_cols].head(10))
print()
print("dtypes:")
print(df[rank_cols].dtypes)

Total rows in df: 1000

Per-column NaN counts:
1      0
2      0
3    120
4    223
5    348
dtype: int64

Rows with ALL 5 filled (your original filter): 652
Rows with AT LEAST 1 filled (the new filter): 1000

First 10 rows of rank_cols, raw:
   1  2    3    4    5
0  1  4  NaN  NaN  NaN
1  1  4  2.0  3.0  0.0
2  1  4  0.0  2.0  3.0
3  1  0  4.0  2.0  3.0
4  4  2  3.0  1.0  0.0
5  1  0  4.0  NaN  NaN
6  4  1  2.0  3.0  0.0
7  3  2  4.0  1.0  0.0
8  1  0  4.0  2.0  3.0
9  4  1  2.0  3.0  0.0

dtypes:
1      int64
2      int64
3    float64
4    float64
5    float64
dtype: object


In [16]:
rank_cols = ['1', '2', '3', '4','5']
df_clean = df[df[rank_cols].notna().any(axis=1)] 

def clean_ballot(row):
    ranked = []
    for c in row:
        if pd.isna(c):
            break
        ranked.append(int(c))
    return ranked

ballots = [(clean_ballot(row), 1) for row in df_clean[rank_cols].values.tolist()]

candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        candidates.add(candidate)
candidates = list(candidates)

print(f"Number of Ballots: {len(ballots)}")
print(f"Candidates: {candidates}")


Number of Ballots: 1000
Candidates: [0, 1, 2, 3, 4]


In [17]:
print(df['1'].dtype)
print(df['1'].unique())
print((df['1'] == 0).sum())
print(df['1'].value_counts(dropna=False))

int64
[1 4 3 2 0]
52
4    338
3    308
1    240
2     62
0     52
Name: 1, dtype: int64


In [18]:
last_column = df.columns[5]
number_counts = df[last_column].value_counts()

print(f"Number of unique values in the last column: {len(number_counts)}")
print(number_counts)

Number of unique values in the last column: 2
0.0    489
3.0    163
Name: 5, dtype: int64


In [19]:
from itertools import combinations
candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        candidates.add(candidate)
candidates = list(candidates)
print(len(candidates), candidates)
def simulate_copeland_fast(ballots, candidates):
    cand_index = {c: i for i, c in enumerate(candidates)}
    n = len(candidates)
    matrix = np.zeros((n, n))
    
    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_index]
        unranked = [c for c in candidates if c not in ranked]
        
        # Ranked candidates vs each other: order on the ballot determines the win
        for c1, c2 in combinations(ranked, 2):
            matrix[cand_index[c1]][cand_index[c2]] += weight
        
        # Every ranked candidate beats every unranked candidate
        for c1 in ranked:
            for c2 in unranked:
                matrix[cand_index[c1]][cand_index[c2]] += weight
        
        # Unranked vs unranked: no preference was expressed, so skip (tie)
    
    scores = {c: 0 for c in candidates}
    for c1, c2 in combinations(candidates, 2):
        i, j = cand_index[c1], cand_index[c2]
        if matrix[i][j] > matrix[j][i]:
            scores[c1] += 1
            scores[c2] -= 1
        elif matrix[j][i] > matrix[i][j]:
            scores[c2] += 1
            scores[c1] -= 1
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\nCopeland Scores:")
    for candidate, score in sorted_scores:
        print(f"  {candidate}: {score}")
    winner = sorted_scores[0][0]
    print(f"\nCopeland Winner: {winner}")
    return winner

5 [0, 1, 2, 3, 4]


In [20]:
copeland_winner = simulate_copeland_fast(ballots, candidates)


Copeland Scores:
  4: 4
  2: 2
  3: 0
  1: -2
  0: -4

Copeland Winner: 4


In [21]:
def simulate_actual_plurality_veto(ballots, candidates):
    cand_set = set(candidates)

    scores = {c: 0 for c in candidates}
    expanded_voters = []

    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_set]
        if ranked:
            scores[ranked[0]] += weight
            for _ in range(weight):
                expanded_voters.append(ranked)

    standing = set(candidates)
    elimination_order = []

    for ranked_ballot in expanded_voters:
        if len(standing) == 1:
            break

        ranked_set = set(ranked_ballot)
        # Every standing candidate this voter left off their ballot is treated as equally "last" and gets vetoed simultaneously.
        unranked_standing = standing - ranked_set

        vetoed_this_pass = set(unranked_standing)

        if not unranked_standing:
            for c in reversed(ranked_ballot):
                if c in standing:
                    vetoed_this_pass = {c}
                    break

        for vetoed in vetoed_this_pass:
            scores[vetoed] -= 1
            if scores[vetoed] <= 0 and vetoed in standing:
                standing.remove(vetoed)
                elimination_order.append(vetoed)

    if standing:
        winner = list(standing)[0]
    else:
        winner = elimination_order[-1]
    print(f"Elimination order: {elimination_order}")
    print(f"Plurality Veto Winner: {winner}")
    return winner, elimination_order

In [22]:
plurality_veto_winner, plurality_veto_elimination = simulate_actual_plurality_veto(ballots, candidates)

Elimination order: [0, 1, 3, 2]
Plurality Veto Winner: 4


In [23]:
last_place_counts = {
    0 : 20,
    1 : 0,
    2 : 8,
    3 : 0,
    4 : 0
}
def simulate_actual_plurality_veto(ballots, candidates, last_place_counts):
    cand_set = set(candidates)

    scores = {c: 0 for c in candidates}
    expanded_voters = []

    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_set]
        if ranked:
            scores[ranked[0]] += weight
            for _ in range(weight):
                expanded_voters.append(ranked)

    # Derive the denominator from the counts themselves 
    # total last-place appearances across all candidates = total complete-ballot weight
    total_last_place = sum(last_place_counts.get(c, 0) for c in candidates)

    if total_last_place > 0:
        last_place_pct = {
            c: last_place_counts.get(c, 0) / total_last_place
            for c in candidates
        }
    else:
        last_place_pct = {c: 0.0 for c in candidates}

    standing = set(candidates)
    elimination_order = []

    for ranked_ballot in expanded_voters:
        if len(standing) == 1:
            break

        ranked_set = set(ranked_ballot)
        unranked_standing = standing - ranked_set

        vetoed_this_pass = set(unranked_standing)

        if not unranked_standing:
            for c in reversed(ranked_ballot):
                if c in standing:
                    vetoed_this_pass = {c}
                    break

        newly_eliminated = []
        for vetoed in vetoed_this_pass:
            scores[vetoed] -= 1
            if scores[vetoed] <= 0 and vetoed in standing:
                newly_eliminated.append(vetoed)

        newly_eliminated.sort(key=lambda c: (-last_place_pct[c], c))

        for c in newly_eliminated:
            if c in standing:
                standing.remove(c)
                elimination_order.append(c)

    if standing:
        winner = list(standing)[0]
    else:
        winner = elimination_order[-1]
    print(f"Elimination order: {elimination_order}")
    print(f"Plurality Veto Winner: {winner}")
    return winner, elimination_order

In [24]:
plurality_veto_winner, plurality_veto_elimination = simulate_actual_plurality_veto(ballots, candidates, last_place_counts)

Elimination order: [0, 1, 3, 2]
Plurality Veto Winner: 4
